In [1]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np  # For defining the ideal arrays (convert to jnp later)
import csv
from jax import lax
from typing import Dict, List, Tuple


# Your ideal ground truth (fixed N_A=8, N_B=8, N_C=16)
def get_ideal_coords() -> dict[str, jnp.ndarray]:
    array_A = np.array([
        [63., 0., 0.],
        [44.55, 44.55, 0.],
        [0., 63., 0.],
        [-44.55, 44.55, 0.],
        [-63., 0., 0.],
        [-44.55, -44.55, 0.],
        [-0., -63., 0.],
        [44.55, -44.55, 0.]
    ])
    array_B = np.array([
        [63., 0., -38.5],
        [44.55, 44.55, -38.5],
        [0., 63., -38.5],
        [-44.55, 44.55, -38.5],
        [-63., 0., -38.5],
        [-44.55, -44.55, -38.5],
        [-0., -63., -38.5],
        [44.55, -44.55, -38.5]
    ])
    array_C = np.array([
        [47.00, 0.00, -68.50],
        [79.00, 0.00, -68.50],
        [55.86, 55.86, -68.50],
        [33.23, 33.23, -68.50],
        [0.00, 47.00, -68.50],
        [0.00, 79.00, -68.50],
        [-55.86, 55.86, -68.50],
        [-33.23, 33.23, -68.50],
        [-47.00, 0.00, -68.50],
        [-79.00, 0.00, -68.50],
        [-55.86, -55.86, -68.50],
        [-33.23, -33.23, -68.50],
        [0.00, -47.00, -68.50],
        [0.00, -79.00, -68.50],
        [55.86, -55.86, -68.50],
        [33.23, -33.23, -68.50],
    ])
    return {
        'A': jnp.array(array_A),  # slight offset to avoid zero distances
        'B': jnp.array(array_B),
        'C': jnp.array(array_C)
    }

# Generalized function to generate random coords for arbitrary N_A, N_B, N_C
def generate_random_coords(
    key: jax.Array,
    n_dict: dict[str, int],
    minval: float = -200.0,
    maxval: float = 200.0
) -> dict[str, jnp.ndarray]:
    coords = {}
    subkeys = jax.random.split(key, len(n_dict))
    for i, (label, n) in enumerate(n_dict.items()):
        coords[label] = jax.random.uniform(subkeys[i], shape=(n, 3), minval=minval, maxval=maxval)
    return coords

def generate_synthetic_data():
    # For coordinates of A-B and B-C pairs, we generate a random set of points sampled from a Gaussian distribution, 
    # centered at the ideal distance value, with a noise parameter (stddev) to control the spread, and this is nice 
    # because later on, I can infer the noise parameter during MCMC sampling and then check if it matches the ground truth used to generate the data.
    # For A-B distances, ideal is 38.5, and I want a noise of around 4.0 angstroms, sample from a Gaussian 
    key = random.PRNGKey(0)
    n_AB_data = 50
    ab_ideal_dist = 38.5
    ab_noise = 4.0
    ab_distances = random.normal(key, shape=(n_AB_data,)) * ab_noise + ab_ideal_dist
    
    # Now for B-C distances, ideal is 34.0, noise around 7.0 angstroms because there are more C particles than B particles and therefore I would 
    # expect more variability in distances measured.
    key, subkey = random.split(key)
    n_BC_data = 80
    bc_ideal_dist = 34.0
    bc_noise = 7.0
    bc_distances = random.normal(subkey, shape=(n_BC_data,)) * bc_noise + bc_ideal_dist
    
    # save all the distances into a csv file or in the form of a h5py file or else a json file, perhaps a csv file
    print("Generated AB distances:", ab_distances)
    print("Generated BC distances:", bc_distances)
    
    with open('synthetic_distances.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['AB_distances'])
        for dist in ab_distances:
            writer.writerow([dist])
        writer.writerow(['BC_distances'])
        for dist in bc_distances:
            writer.writerow([dist])
    
    return ab_distances, bc_distances
    
# Generalized all-vs-all distance matrix
# Input: coords dict with shapes {'A': (N_A, 3), 'B': (N_B, 3), 'C': (N_C, 3)}
# Output: (N_total, N_total) where order is all A's, then B's, then C's
def compute_pair_distance_matrices(
    coords: Dict[str, jnp.ndarray], 
    pair_types: List[str]
) -> Dict[str, jnp.ndarray]:
    """
    Compute distance matrices for each pair type.
    
    Args:
        coords: Dictionary with keys 'A', 'B', 'C' and values of shape (N_i, 3)
        pair_types: List of pair type strings like ['AA', 'AB', 'BC']
    
    Returns:
        Dictionary mapping pair_type -> distance matrix of shape (N_i, N_j)
    """
    distance_matrices = {}
    
    for pair_type in pair_types:
        type1, type2 = pair_type[0], pair_type[1]
        
        coords1 = coords[type1]  # Shape: (N_i, 3)
        coords2 = coords[type2]  # Shape: (N_j, 3)
        
        # Compute all pairwise distances
        diff = coords1[:, jnp.newaxis, :] - coords2[jnp.newaxis, :, :]
        dist_matrix = jnp.linalg.norm(diff, axis=-1)  # Shape: (N_i, N_j)
        
        distance_matrices[pair_type] = dist_matrix
    
    return distance_matrices


@jax.jit
def compute_nll_single_pair_type(
    dist_matrix: jnp.ndarray,
    d0: float,
    sigma: float,
    is_same_type: bool
) -> Tuple[jnp.ndarray, float]:
    """
    Compute NLL matrix and select best pairs for a single pair type.
    
    Args:
        dist_matrix: Distance matrix of shape (N_i, N_j)
        d0: Expected distance
        sigma: Distance uncertainty
        is_same_type: Whether this is AA, BB, or CC (vs AB, AC, BC)
    
    Returns:
        nll_matrix: Negative log-likelihood matrix (N_i, N_j)
        total_nll: Sum of NLL for selected pairs
    """
    # Compute NLL matrix (vectorized)
    nll_matrix = (dist_matrix - d0)**2 / (2 * sigma**2) + jnp.log(sigma * jnp.sqrt(2 * jnp.pi))
    
    # Define branches for same-type vs different-type
    def same_type_branch(operands):
        nll_mat, _ = operands
        n_rows, n_cols = nll_mat.shape
        # For same-type, only use upper triangle (assumes square matrix)
        mask = jnp.triu(jnp.ones((n_rows, n_cols), dtype=bool), k=1)
        masked_nll = jnp.where(mask, nll_mat, jnp.inf)
        
        # Row-wise minima
        row_mins = jnp.min(masked_nll, axis=1)  # (N_i,)
        
        # Column-wise minima
        col_mins = jnp.min(masked_nll, axis=0)  # (N_j,)
                
        # Sum the selected scores
        total_nll = jnp.sum(jnp.where(jnp.isfinite(row_mins), row_mins, 0.0)) + \
                    jnp.sum(jnp.where(jnp.isfinite(col_mins), col_mins, 0.0))
        
        # Divide by 2 to account for potential double-counting
        return total_nll / 2.0

    def different_type_branch(operands):
        nll_mat, _ = operands
        # Row-wise minima
        row_mins = jnp.min(nll_mat, axis=1)  # (N_i,)
        
        # Column-wise minima
        col_mins = jnp.min(nll_mat, axis=0)  # (N_j,)
        
        # Sum both (union strategy)
        return jnp.sum(row_mins) + jnp.sum(col_mins)
    
    # Use lax.cond for JAX-compatible branching
    total_nll = lax.cond(
        is_same_type,
        same_type_branch,
        different_type_branch,
        (nll_matrix, dist_matrix)
    )
    
    return nll_matrix, total_nll


def compute_total_pair_nll_jax(
    coords: Dict[str, jnp.ndarray],
    pair_distances: Dict[str, float],
    nuisance_parameters: Dict[str, float]
) -> float:
    """
    JAX-compatible version of PairNLL.compute_score()
    
    Args:
        coords: Dictionary {'A': (N_A, 3), 'B': (N_B, 3), 'C': (N_C, 3)}
        pair_distances: Expected distances {'AA': d0_AA, 'AB': d0_AB, ...}
        nuisance_parameters: Sigmas {'AA': sigma_AA, 'AB': sigma_AB, ...}
    
    Returns:
        Total negative log-likelihood
    """
    pair_types = list(nuisance_parameters.keys())
    
    # Step 1: Compute all distance matrices
    distance_matrices = compute_pair_distance_matrices(coords, pair_types)
    
    # Step 2: Compute NLL for each pair type
    total_nll = 0.0
    
    for pair_type in pair_types:
        dist_matrix = distance_matrices[pair_type]
        d0 = pair_distances[pair_type]
        sigma = nuisance_parameters[pair_type]
        
        is_same_type = (pair_type[0] == pair_type[1])
        
        _, pair_nll = compute_nll_single_pair_type(
            dist_matrix, d0, sigma, is_same_type
        )
        
        total_nll += pair_nll
    
    return total_nll


# Example usage:
if __name__ == "__main__":
    import jax.random as random
    
    key = random.PRNGKey(0)
    
    # find the score for the ideal structure 
    coords = get_ideal_coords()
    
    pair_distances = {'AA': 48.22, 'AB': 38.5, 'BC': 34.0}
    nuisance_parameters = {'AA': 1.0, 'AB': 1.0, 'BC': 1.0}
    
    # Compute distance matrices
    dist_mats = compute_pair_distance_matrices(coords, ['AA', 'AB', 'BC'])
    
    # Compute total NLL
    total_nll = compute_total_pair_nll_jax(coords, pair_distances, nuisance_parameters)
    print(f"\nTotal NLL: {total_nll:.4f}")

W0000 00:00:1767744168.540245  243241 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1767744168.671753  243241 service.cc:145] XLA service 0x11913ad20 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767744168.671773  243241 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1767744168.674751  243241 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1767744168.674760  243241 mps_client.cc:384] XLA backend will use up to 5726240768 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M2

systemMemory: 8.00 GB
maxCacheSize: 2.67 GB



XlaRuntimeError: UNIMPLEMENTED: default_memory_space is not supported.

In [155]:
import jax
import jax.numpy as jnp
import jax.random as random
import blackjax
from functools import partial
from typing import Dict, Tuple
import h5py
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# Particle counts
# ------------------------------------------------------------
N_A, N_B, N_C = 8, 8, 16

def flatten_coords(coords: Dict[str, jnp.ndarray]) -> jnp.ndarray:
    """Turn dict of coords into one long vector."""
    return jnp.concatenate([coords['A'].ravel(),
                            coords['B'].ravel(),
                            coords['C'].ravel()])

def unflatten_coords(flat: jnp.ndarray) -> Dict[str, jnp.ndarray]:
    """Turn flat vector back into dict of coords."""
    nA = N_A * 3
    nB = N_B * 3
    A = flat[:nA].reshape(N_A, 3)
    B = flat[nA:nA+nB].reshape(N_B, 3)
    C = flat[nA+nB:].reshape(N_C, 3)
    return {'A': A, 'B': B, 'C': C}
# ------------------------------------------------------------
# Sigma MCMC
# ------------------------------------------------------------
SIGMA_LABELS = ("AA", "AB", "BC")
LOG_SIGMA_RANGE = (jnp.log(0.1), jnp.log(20.0))

def log_prior_logsigmas(log_sigmas: jnp.ndarray) -> jnp.ndarray:
    """Uniform prior in log-space."""
    low, high = LOG_SIGMA_RANGE
    inside = jnp.all((log_sigmas >= low) & (log_sigmas <= high))
    return jnp.where(inside, 0.0, -jnp.inf)

def log_prob_logsigmas(
    log_sigmas: jnp.ndarray,
    coords: Dict[str, jnp.ndarray],
    target_dists: Dict[str, float],
) -> jnp.ndarray:
    """Posterior log-probability for log-sigmas (coords fixed)."""
    sigma_vals = jnp.exp(log_sigmas)
    sigmas = {label: sigma_vals[i] for i, label in enumerate(SIGMA_LABELS)}
    nll = compute_total_pair_nll_jax(coords, target_dists, sigmas)
    log_prior = log_prior_logsigmas(log_sigmas)
    return -nll + log_prior  # Higher is better

def run_sigma_mcmc(
    key: jax.Array,
    coords: Dict[str, jnp.ndarray],
    target_dists: Dict[str, float],
    init_sigmas: jnp.ndarray,
    n_steps: int = 50_000,
    step_sizes: jnp.ndarray = jnp.array([0.05, 0.05, 0.05]),
    save_every: int = 50,
    output_file: str = "sigma_chain.h5",
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """Random-walk Metropolis over log-sigmas."""
    logprob = partial(log_prob_logsigmas, coords=coords, target_dists=target_dists)
    initial_log_sigmas = jnp.log(init_sigmas)
    proposal = blackjax.mcmc.random_walk.normal(step_sizes)
    rmh = blackjax.rmh(logprob, proposal)
    state = rmh.init(initial_log_sigmas)

    @jax.jit
    def one_step(state, key):
        new_state, info = rmh.step(key, state)
        return new_state, (new_state.position, new_state.logdensity, info.is_accepted)

    keys = random.split(key, n_steps)
    chain, log_probs, accepts = [], [], []

    print(f"Running {n_steps:,} sigma-MCMC steps...")
    for i, k in enumerate(keys):
        state, (log_sigma, lp, acc) = one_step(state, k)
        accepts.append(acc)

        if i % save_every == 0:
            chain.append(log_sigma)
            log_probs.append(lp)

        if i % 5000 == 0:
            recent = np.mean(accepts[-100:]) if len(accepts) >= 100 else np.mean(accepts)
            sigmas_now = jnp.exp(log_sigma)
            print(
                f"  Step {i:6d} | log_prob: {lp:8.2f} "
                f"| sigmas: {sigmas_now.tolist()} | accept: {recent:.1%}"
            )

    chain = jnp.stack(chain)
    log_probs = jnp.array(log_probs)
    acceptance = np.mean(accepts)
    print(f"\nDone! Acceptance rate: {acceptance:.1%}")
    print(f"Saved {len(chain):,} draws (every {save_every} steps)")

    save_sigma_chain(chain, log_probs, target_dists, output_file)
    return chain, log_probs

def save_sigma_chain(
    log_sigma_chain: jnp.ndarray,
    log_probs: jnp.ndarray,
    target_dists: Dict[str, float],
    filename: str,
):
    """Persist sigma samples + metadata."""
    sigma_chain = np.exp(np.array(log_sigma_chain))
    with h5py.File(filename, "w") as f:
        f.attrs["creation_date"] = datetime.now().isoformat()
        f.create_dataset("log_sigmas", data=np.array(log_sigma_chain), compression="gzip")
        f.create_dataset("sigmas", data=sigma_chain, compression="gzip")
        f.create_dataset("log_probabilities", data=np.array(log_probs), compression="gzip")
        params = f.create_group("target_distances")
        for k, v in target_dists.items():
            params.attrs[k] = v
        best = np.argmax(log_probs)
        f.attrs["best_log_prob"] = float(log_probs[best])
        f.attrs["best_sigmas"] = sigma_chain[best].tolist()
    print(f"Saved sigma chain to: {filename}")

if __name__ == "__main__":
    coords = get_ideal_coords()  # Fixed structure
    target_distances = {"AA": 48.22, "AB": 38.5, "BC": 34.0}
    init_sigma_guess = jnp.array([5.0, 5.0, 5.0])

    sigma_chain, log_probs = run_sigma_mcmc(
        key=random.PRNGKey(123),
        coords=coords,
        target_dists=target_distances,
        init_sigmas=init_sigma_guess,
        n_steps=100_000,
        step_sizes=jnp.array([0.05, 0.05, 0.05]),  # tune as needed
        save_every=50,
        output_file="sigma_chain.h5",
    )

Running 100,000 sigma-MCMC steps...
  Step      0 | log_prob:   -43.04 | sigmas: [1.0555592775344849, 1.0193930864334106, 0.9658486247062683] | accept: 100.0%
  Step   5000 | log_prob:   -38.32 | sigmas: [0.943476140499115, 0.881650447845459, 0.9029766917228699] | accept: 10.0%
  Step  10000 | log_prob:   -39.61 | sigmas: [1.0459275245666504, 0.959138035774231, 0.8743098974227905] | accept: 37.0%
  Step  15000 | log_prob:   -39.85 | sigmas: [0.9483098983764648, 0.9379358291625977, 0.9221618175506592] | accept: 27.0%
  Step  20000 | log_prob:   -43.95 | sigmas: [1.0952681303024292, 1.0173598527908325, 0.9937373995780945] | accept: 30.0%
  Step  25000 | log_prob:   -39.70 | sigmas: [0.9902347922325134, 0.9450504779815674, 0.9002611637115479] | accept: 33.0%
  Step  30000 | log_prob:   -41.44 | sigmas: [1.0109233856201172, 0.9740093946456909, 0.9432651996612549] | accept: 34.0%
  Step  35000 | log_prob:   -39.74 | sigmas: [0.8718147277832031, 0.9124361276626587, 0.9581081867218018] | acce

In [ ]:
#-------------------------------------------------------------------------------------------------------
# Now, consider a couple of particles, for example the system ABCC and then put this system inside a box
# Now we are going to do simulation where we not only sample the sigmas but also the 
# coordinates of the particles, i.e., do joint sampling of coordinates and sigmas
# -------------------------------------------------------------------------------------------------------
import time
# generate random key
key = random.PRNGKey(56)
coords = generate_random_coords(key=key, n_dict={'A':2, 'B':2, 'C':4}, minval=-50.0, maxval=50.0)

@jax.jit
def compute_distance_matrix(coords: Dict[str, jnp.ndarray]) -> jnp.ndarray:
    """Compute distance matrix between all particle pairs."""
    all_coords = jnp.concatenate([coords['A'], coords['B'], coords['C']], axis=0)  # (N_total, 3)
    diff = all_coords[:, jnp.newaxis, :] - all_coords[jnp.newaxis, :, :]  # (N_total, N_total, 3)
    dist_matrix = jnp.linalg.norm(diff, axis=-1)  # (N_total, N_total)
    
    return dist_matrix
    
def log_excluded_volume(coords: Dict[str, jnp.ndarray], radius: Dict[str, float]) -> float:
    """Compute log excluded volume penalty for coords in a box.
       When the distance between any two particles is less than the sum of their radii,
       apply a penalty. The penalty is of the form (d_min - d_ij)^2*sigma where d_min is 
       the sum of the radii."""
    
    dismat = compute_distance_matrix(coords)
    # Create a radius matrix 
    ordered_particles = []
    for key, elements in coords.items():
        N = elements.shape[0]
        # Create a list with the same order of the keys
        ordered_particles.extend([key]*N)
        print(f"Number of particles of type {key}: {N}")
    
    # Create sum radius matrix
    # Create a jax array of shape N where each element is the radius of the corresponding particle
    radius_array = jnp.array([radius[p] for p in ordered_particles])
    a_col = radius_array[:, jnp.newaxis]
    b_col = radius_array[jnp.newaxis, :]
    radius_sum_matrix = a_col + b_col  # shape (N, N)
    
    print("Distance matrix:\n", dismat)
    
    diffmat = dismat - radius_sum_matrix
    
    upp_tri = jnp.triu(diffmat, k=1)
    print("Upper triangle of difference matrix:\n", upp_tri)
    
    # Check if there are any negative values in the upper triangle
    penalty_mask = upp_tri < 0.0
    # square the negative values and sum them up
    penalty_values = jnp.where(penalty_mask, upp_tri**2, 0.0)
    penalty = jnp.sum(penalty_values)
    print(f"Excluded volume penalty: {penalty}")
    
    return -penalty  # log probability is negative penalty
        
# Calcualte the Negative log likelihood for these random coordinates
pair_distances = {'AA': 48.22, 'AB': 38.5, 'BC': 34.0}
radius = {'A': 24.0, 'B': 14.0, 'C': 16.0}
nuisance_parameters = {'AA': 1.0, 'AB': 1.0, 'BC': 1.0}
total_nll = compute_total_pair_nll_jax(coords, pair_distances, nuisance_parameters)
print(f"\nTotal NLL for random coords: {total_nll:.4f}")

dismat = compute_distance_matrix(coords)
print("Distance matrix:\n", dismat)
%timeit compute_distance_matrix(coords)

print("Computing log excluded volume penalty...", log_excluded_volume(coords, radius))

time_start = time.time()
log_excluded_volume(coords, radius)
time_end = time.time()
print(f"Computation Time for log excluded volume: {time_end - time_start:.6f} seconds")

def log_probability(coords_flat, params):
    '''
    Total log probability = sum of all terms 
    '''
    coords = unflatten_coords(coords_flat)

Randomly generated coordinates: {'A': Array([[ 38.929283 ,  -4.014814 ,   2.2464871],
       [-44.898426 , -32.236004 ,  29.063904 ]], dtype=float32), 'B': Array([[-25.851095,  29.7503  ,  26.653326],
       [-19.612778,  10.680735,  35.197674]], dtype=float32), 'C': Array([[ 42.703735 , -26.145613 , -13.694704 ],
       [-12.119495 , -43.98061  ,  25.311756 ],
       [ -8.245397 , -33.108925 ,  32.799973 ],
       [-10.964084 , -43.99041  ,   0.7688999]], dtype=float32)}

Total NLL for random coords: 4708.6484
Distance matrix:
 [[ 0.       92.426704 77.02126  68.76709  27.534348 68.81315  63.28849
  63.94982 ]
 [92.426704  0.       64.891556 50.187984 97.67052  35.02103  36.853287
  45.719936]
 [77.02126  64.891556  0.       21.807587 97.22182  75.0107   65.56694
  79.557014]
 [68.76709  50.187984 21.807587  0.       87.34983  56.051258 45.30453
  65.18492 ]
 [27.534348 97.67052  97.22182  87.34983   0.       69.60731  69.32573
  58.37694 ]
 [68.81315  35.02103  75.0107   56.051258 69

In [145]:
import jax
import jax.numpy as jnp
from typing import Dict

# 1. Helper to flatten coords (run this once or inside the JIT wrapper)
def get_flat_coords_and_radii(coords: Dict[str, jnp.ndarray], radii: Dict[str, float]):
    """
    Concatenates coordinates and creates a corresponding radius vector.
    This prepares data for the JIT-compiled kernel.
    """
    # Concatenate all coordinate arrays (Order matters! maintain A, then B, then C...)
    # Sorting keys ensures deterministic order
    keys = sorted(coords.keys()) 
    flat_coords = jnp.concatenate([coords[k] for k in keys], axis=0)
    
    # Create the radius vector matching the order of coords
    radius_list = []
    for k in keys:
        n_particles = coords[k].shape[0]
        # Create array of shape (N,) filled with the radius value
        radius_list.append(jnp.full((n_particles,), radii[k]))
        
    flat_radii = jnp.concatenate(radius_list, axis=0)
    
    return flat_coords, flat_radii

# 2. The JIT-compiled Calculation Kernel
@jax.jit
def log_excluded_volume_kernel(flat_coords: jnp.ndarray, flat_radii: jnp.ndarray, k_stiffness: float = 100.0) -> float:
    """
    Efficiently computes excluded volume penalty.
    
    Args:
        flat_coords: Shape (N_total, 3)
        flat_radii: Shape (N_total,)
        k_stiffness: Spring constant for the repulsion
    """
    # 1. Compute Distance Matrix (N, N)
    # Using (x-y)^2 trick or explicit broadcasting
    diff = flat_coords[:, None, :] - flat_coords[None, :, :]
    # Add epsilon to avoid sqrt(0) gradients if needed, though for repulsion usually fine
    dists = jnp.sqrt(jnp.sum(diff**2, axis=-1) + 1e-6)
    
    # 2. Compute Sum of Radii Matrix (N, N)
    # r_sum[i, j] = r[i] + r[j]
    r_sum_matrix = flat_radii[:, None] + flat_radii[None, :]
    
    # 3. Calculate Overlap
    # Overlap = (r_i + r_j) - d_ij
    # If d_ij > sum_radii, this is negative (no overlap).
    # If d_ij < sum_radii, this is positive (overlap).
    overlaps = r_sum_matrix - dists
    
    # 4. Filter for only positive overlaps (actual collisions)
    # We ignore self-interactions (diagonal) naturally because r_sum > 0 and dist=0, 
    # BUT we usually want to mask the diagonal explicitly to avoid self-clashing logic errors.
    # However, physically, a particle overlaps with itself? No, we ignore i==j.
    
    # Mask out the diagonal (dists=0) to prevent self-repulsion
    # or strictly use upper triangle.
    
    # Use ReLU: max(0, overlap). 
    # This automatically zeros out non-overlapping pairs.
    active_overlaps = jax.nn.relu(overlaps)
    
    # 5. Mask the diagonal / lower triangle to avoid double counting
    # We multiply by the upper triangle mask (k=1 excludes diagonal)
    N = flat_coords.shape[0]
    mask = jnp.triu(jnp.ones((N, N), dtype=bool), k=1)
    
    # Apply mask
    final_overlaps = jnp.where(mask, active_overlaps, 0.0)
    
    # 6. Compute Penalty (Square the overlaps)
    # Penalty = sum( k * overlap^2 )
    penalty = jnp.sum(k_stiffness * final_overlaps**2)
    
    # Return negative penalty (log probability)
    return penalty

def log_probability(coords_flat: jnp.ndarray, coords: Dict[str, jnp.ndarray], radii_flat: jnp.ndarray, 
                    target_dists: Dict[str, float], nuisance_parameters: Dict[str, float]) -> float:
    """
    Wrapper to compute total log probability given flat coords and radii.
    
    Returns negative of (NLL + excluded volume penalty) so that:
    - Higher log_prob = better configuration
    - MCMC will maximize this (minimize NLL + penalty)
    """
    # Excluded volume penalty (positive when overlapping)
    ev_penalty = log_excluded_volume_kernel(coords_flat, radii_flat)
    
    # NLL (positive value)
    nll = compute_total_pair_nll_jax(
        coords,
        pair_distances=target_dists,
        nuisance_parameters=nuisance_parameters
    )
    
    # Log probability = negative of total cost
    # Higher is better for MCMC
    return (nll + ev_penalty)

# --- Usage Example ---

# Define data
coords = generate_random_coords(
    key=random.PRNGKey(56),
    n_dict={'A': 2, 'B': 2, 'C': 4},
    minval=-150.0,
    maxval=150.0
)
radii = {'A': 24.0, 'B': 14.0, 'C': 16.0}

# Pre-processing (Can be done inside the model step if shapes don't change)
flat_coords, flat_radii = get_flat_coords_and_radii(coords, radii)

# Run Calculation
time_start = time.time()
log_prob = log_excluded_volume_kernel(flat_coords, flat_radii)
time_end = time.time()

target_dists = {'AA': 48.22, 'AB': 38.5, 'BC': 34.0}
nuisance_parameters = {'AA': 5.0, 'AB': 3.0, 'BC': 4.0}

log_prob_total = log_probability(flat_coords, coords, flat_radii, target_dists, nuisance_parameters)
print("Total Log Probability:", log_prob_total)



Total Log Probability: 9868.119


In [138]:
import blackjax

def setup_coordinate_rmh(log_prob_fn, step_size):
    """
    Set up Random Walk Metropolis for coordinate sampling.
    
    step_size: scalar or array matching coordinate dimensions
    """
    # Isotropic Gaussian proposal
    proposal_distribution = blackjax.mcmc.random_walk.normal(step_size)
    
    # Build the RMH kernel
    rmh_kernel = blackjax.rmh(log_prob_fn, proposal_distribution)
    
    return rmh_kernel


def run_rmh_chain(key, kernel, initial_position, n_steps):
    """
    Run a single RMH chain.
    
    Returns: positions (n_steps, n_dims), log_probs (n_steps,), acceptance (n_steps,)
    """
    # Initialize state
    state = kernel.init(initial_position)
    
    # Define one step function for scan
    def step_fn(state, key):
        new_state, info = kernel.step(key, state)
        return new_state, (new_state.position, new_state.logdensity, info.is_accepted)
    
    # Generate all random keys upfront
    keys = random.split(key, n_steps)
    
    # Run the chain using scan
    final_state, (positions, log_probs, accepted) = jax.lax.scan(
        step_fn, state, keys
    )
    
    return positions, log_probs, accepted

# Run the code
log_prob_fn = lambda coords_flat: log_probability(
    coords_flat,
    coords,
    flat_radii,
    target_dists,
    nuisance_parameters
)
step_size = 0.5  # Adjust based on scale of coordinates
rmh_kernel = setup_coordinate_rmh(log_prob_fn, step_size)
initial_position = flat_coords
n_steps = 100000
key = random.PRNGKey(789)
positions, log_probs, accepted = run_rmh_chain(key, rmh_kernel, initial_position, n_steps)

In [146]:
import blackjax
import h5py
from datetime import datetime

# Particle counts for this system (must match coords structure)
N_A_small, N_B_small, N_C_small = 2, 2, 4

def unflatten_coords_small(flat: jnp.ndarray) -> Dict[str, jnp.ndarray]:
    """Unflatten for the small system (2A, 2B, 4C)."""
    nA = N_A_small * 3
    nB = N_B_small * 3
    A = flat[:nA].reshape(N_A_small, 3)
    B = flat[nA:nA+nB].reshape(N_B_small, 3)
    C = flat[nA+nB:].reshape(N_C_small, 3)
    return {'A': A, 'B': B, 'C': C}


def setup_coordinate_rmh(log_prob_fn, step_size):
    """
    Set up Random Walk Metropolis for coordinate sampling.
    
    step_size: scalar or array matching coordinate dimensions
    """
    proposal_distribution = blackjax.mcmc.random_walk.normal(step_size)
    rmh_kernel = blackjax.rmh(log_prob_fn, proposal_distribution)
    return rmh_kernel


def run_rmh_chain_with_logging(
    key,
    kernel,
    initial_position,
    n_steps,
    save_every=500,
    print_every=1000,
    output_file="mcmc_coords.h5"
):
    """
    Run RMH chain with progress logging and HDF5 saving.
    
    Args:
        key: JAX random key
        kernel: BlackJAX RMH kernel
        initial_position: Initial flat coordinates
        n_steps: Total MCMC steps
        save_every: Save coordinates every N steps
        print_every: Print progress every N steps
        output_file: HDF5 output filename
    
    Returns:
        saved_positions, saved_log_probs, acceptance_rate
    """
    state = kernel.init(initial_position)
    
    @jax.jit
    def one_step(state, key):
        new_state, info = kernel.step(key, state)
        return new_state, (new_state.position, new_state.logdensity, info.is_accepted)
    
    keys = random.split(key, n_steps)
    
    # Storage
    saved_positions = []
    saved_log_probs = []
    accepts = []
    
    print(f"{'='*70}")
    print(f"Starting MCMC Sampling")
    print(f"{'='*70}")
    print(f"Total steps: {n_steps:,}")
    print(f"Save every: {save_every}")
    print(f"Initial log_prob: {state.logdensity:.4f}")
    print(f"{'='*70}\n")
    
    for i, k in enumerate(keys):
        state, (position, log_prob, accepted) = one_step(state, k)
        accepts.append(float(accepted))
        
        # Save at intervals
        if i % save_every == 0:
            saved_positions.append(np.array(position))
            saved_log_probs.append(float(log_prob))
        
        # Print progress
        if i % print_every == 0:
            recent_accept = np.mean(accepts[-min(1000, len(accepts)):])
            print(
                f"Step {i:7,} / {n_steps:,} | "
                f"log_prob: {log_prob:10.2f} | "
                f"accept: {recent_accept:5.1%}"
            )
    
    # Final stats
    acceptance_rate = np.mean(accepts)
    saved_positions = np.array(saved_positions)
    saved_log_probs = np.array(saved_log_probs)
    
    print(f"\n{'='*70}")
    print(f"MCMC Complete!")
    print(f"{'='*70}")
    print(f"Final log_prob: {log_prob:.4f}")
    print(f"Best log_prob: {np.max(saved_log_probs):.4f}")
    print(f"Overall acceptance: {acceptance_rate:.1%}")
    print(f"Saved {len(saved_positions)} frames")
    
    # Save to HDF5
    save_mcmc_to_hdf5(
        saved_positions,
        saved_log_probs,
        acceptance_rate,
        output_file
    )
    
    return saved_positions, saved_log_probs, acceptance_rate


def save_mcmc_to_hdf5(
    positions: np.ndarray,
    log_probs: np.ndarray,
    acceptance_rate: float,
    filename: str
):
    """
    Save MCMC samples to HDF5 in format compatible with RMF3 converter.
    
    Creates structure:
        coordinates/A: (n_samples, n_A, 3)
        coordinates/B: (n_samples, n_B, 3)
        coordinates/C: (n_samples, n_C, 3)
        log_probabilities: (n_samples,)
        best_configuration/...
    """
    n_samples = positions.shape[0]
    
    # Unflatten all positions
    coords_A = np.zeros((n_samples, N_A_small, 3))
    coords_B = np.zeros((n_samples, N_B_small, 3))
    coords_C = np.zeros((n_samples, N_C_small, 3))
    
    for i in range(n_samples):
        coords_dict = unflatten_coords_small(positions[i])
        coords_A[i] = np.array(coords_dict['A'])
        coords_B[i] = np.array(coords_dict['B'])
        coords_C[i] = np.array(coords_dict['C'])
    
    with h5py.File(filename, 'w') as f:
        # Metadata
        f.attrs['creation_date'] = datetime.now().isoformat()
        f.attrs['n_samples'] = n_samples
        f.attrs['n_A'] = N_A_small
        f.attrs['n_B'] = N_B_small
        f.attrs['n_C'] = N_C_small
        
        # Coordinates group (required by RMF3 converter)
        coords_grp = f.create_group('coordinates')
        coords_grp.create_dataset('A', data=coords_A, compression='gzip')
        coords_grp.create_dataset('B', data=coords_B, compression='gzip')
        coords_grp.create_dataset('C', data=coords_C, compression='gzip')
        
        # Log probabilities
        f.create_dataset('log_probabilities', data=log_probs, compression='gzip')
        
        # Diagnostics
        diag_grp = f.create_group('diagnostics')
        diag_grp.attrs['acceptance_rate'] = acceptance_rate
        diag_grp.attrs['best_log_prob'] = float(np.max(log_probs))
        diag_grp.attrs['final_log_prob'] = float(log_probs[-1])
        
        # Parameters
        param_grp = f.create_group('parameters')
        param_grp.attrs['target_dist_AA'] = target_dists['AA']
        param_grp.attrs['target_dist_AB'] = target_dists['AB']
        param_grp.attrs['target_dist_BC'] = target_dists['BC']
        param_grp.attrs['sigma_AA'] = nuisance_parameters['AA']
        param_grp.attrs['sigma_AB'] = nuisance_parameters['AB']
        param_grp.attrs['sigma_BC'] = nuisance_parameters['BC']
        
        # Best configuration
        best_idx = np.argmax(log_probs)
        best_grp = f.create_group('best_configuration')
        best_grp.attrs['sample_index'] = best_idx
        best_grp.attrs['log_probability'] = float(log_probs[best_idx])
        best_grp.create_dataset('A', data=coords_A[best_idx])
        best_grp.create_dataset('B', data=coords_B[best_idx])
        best_grp.create_dataset('C', data=coords_C[best_idx])
    
    print(f"\nSaved to: {filename}")
    print(f"  Frames: {n_samples}")
    print(f"  Best log_prob: {np.max(log_probs):.2f} (frame {best_idx})")


# ============================================================
# Run the MCMC
# ============================================================

# Log probability function (coords must be defined beforehand)
n_particles = flat_radii.shape[0]  # Total number of particles

log_prob_fn = lambda coords_flat: log_probability(
    coords_flat.reshape(n_particles, 3),  # Reshape 1D -> (N, 3) for excluded volume
    unflatten_coords_small(coords_flat),  # Unflatten to dict for NLL
    flat_radii,
    target_dists,
    nuisance_parameters
)

step_size = 20.0
rmh_kernel = setup_coordinate_rmh(log_prob_fn, step_size)
initial_position = flat_coords.ravel()

# Run with logging
positions, log_probs, acc_rate = run_rmh_chain_with_logging(
    key=random.PRNGKey(789),
    kernel=rmh_kernel,
    initial_position=initial_position,
    n_steps=500_000,
    save_every=500,
    print_every=5000,
    output_file="mcmc_coords.h5"
)

Starting MCMC Sampling
Total steps: 500,000
Save every: 500
Initial log_prob: 9868.1191

Step       0 / 500,000 | log_prob:  117569.18 | accept: 100.0%
Step   5,000 / 500,000 | log_prob:  900438.38 | accept:  0.0%
Step  10,000 / 500,000 | log_prob:  900438.38 | accept:  0.0%
Step  15,000 / 500,000 | log_prob: 1206744.00 | accept:  0.1%
Step  20,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  25,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  30,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  35,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  40,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  45,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  50,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  55,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  60,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  65,000 / 500,000 | log_prob: 1206744.00 | accept:  0.0%
Step  70,000 / 500,000 | log_prob: 1206744

In [154]:
# ============================================================
# FIX: Create proper unflatten function for the system size
# ============================================================

# Add this BEFORE log_probability_from_scores

def create_unflatten_function(n_dict: Dict[str, int]):
    """
    Factory function to create an unflatten function for any system size.
    
    Args:
        n_dict: {'A': n_A, 'B': n_B, 'C': n_C}
    
    Returns:
        unflatten function
    """
    n_A = n_dict['A']
    n_B = n_dict['B']
    n_C = n_dict['C']
    
    def unflatten_coords(flat: jnp.ndarray) -> Dict[str, jnp.ndarray]:
        nA = n_A * 3
        nB = n_B * 3
        A = flat[:nA].reshape(n_A, 3)
        B = flat[nA:nA+nB].reshape(n_B, 3)
        C = flat[nA+nB:].reshape(n_C, 3)
        return {'A': A, 'B': B, 'C': C}
    
    return unflatten_coords


# ============================================================
# UPDATED: log_probability_from_scores with flexible unflatten
# ============================================================

def log_probability_from_scores(
    coords_flat: jnp.ndarray,
    flat_radii: jnp.ndarray,
    target_dists: Dict[str, float],
    nuisance_parameters: Dict[str, float],
    unflatten_fn,  # NEW: pass the unflatten function
    k_ev: float = 1.0
) -> float:
    """
    Convert IMP-style scores (lower=better) to log probability (higher=better).
    
    Args:
        coords_flat: Flattened coordinates (N*3,)
        flat_radii: Radii for each particle (N,)
        target_dists: Target distances {'AA': d0, ...}
        nuisance_parameters: Sigmas {'AA': σ, ...}
        unflatten_fn: Function to unflatten coords
        k_ev: Weight for excluded volume penalty
    
    Returns:
        log_probability (higher = better)
    """
    n_particles = flat_radii.shape[0]
    coords_2d = coords_flat.reshape(n_particles, 3)
    
    # Unflatten for NLL computation
    coords_dict = unflatten_fn(coords_flat)
    
    # 1. Data term: NLL (POSITIVE, larger = worse fit)
    nll_data = compute_total_pair_nll_jax(
        coords_dict,
        pair_distances=target_dists,
        nuisance_parameters=nuisance_parameters
    )
    
    # 2. Excluded volume penalty (POSITIVE when overlapping)
    ev_penalty = log_excluded_volume_kernel(coords_2d, flat_radii, k_stiffness=k_ev)
    
    # 3. Total IMP-style score (lower = better)
    total_score = nll_data + ev_penalty
    
    # 4. Convert to log probability for MCMC (higher = better)
    log_prob = -total_score
    
    return log_prob


# ============================================================
# Verify the math with your ideal structure
# ============================================================

print("="*70)
print("VERIFYING SCORE CONVERSION")
print("="*70)

# Test with ideal coords (LARGE SYSTEM: 8A, 8B, 16C)
ideal_coords = get_ideal_coords()
ideal_flat, ideal_radii = get_flat_coords_and_radii(ideal_coords, radii)

# Create unflatten function for this system size
unflatten_ideal = create_unflatten_function({'A': 8, 'B': 8, 'C': 16})

print(f"\nSystem size: {ideal_flat.shape[0]} particles ({ideal_flat.ravel().shape[0]} coordinates)")

# Test with tight sigma (σ=1)
tight_nuisance = {'AA': 1.0, 'AB': 1.0, 'BC': 1.0}

ideal_log_prob = log_probability_from_scores(
    ideal_flat.ravel(),
    ideal_radii,
    target_dists={'AA': 48.22, 'AB': 38.5, 'BC': 34.0},
    nuisance_parameters=tight_nuisance,
    unflatten_fn=unflatten_ideal
)

print(f"\nIdeal structure (σ=1):")
print(f"  log_prob = {ideal_log_prob:.2f}")
print(f"  (Should be a SMALL negative number, like -10 to -50)")

# Test with random coords (SMALL SYSTEM: 2A, 2B, 4C)
random_coords = generate_random_coords(
    key=random.PRNGKey(999),
    n_dict={'A': 2, 'B': 2, 'C': 4},
    minval=-200.0,
    maxval=200.0
)
random_flat, random_radii = get_flat_coords_and_radii(random_coords, radii)

# Create unflatten function for small system
unflatten_small = create_unflatten_function({'A': 2, 'B': 2, 'C': 4})

print(f"\nSmall system: {random_flat.shape[0]} particles ({random_flat.ravel().shape[0]} coordinates)")

random_log_prob = log_probability_from_scores(
    random_flat.ravel(),
    random_radii,
    target_dists={'AA': 48.22, 'AB': 38.5, 'BC': 34.0},
    nuisance_parameters=tight_nuisance,
    unflatten_fn=unflatten_small
)

print(f"\nRandom structure (σ=1):")
print(f"  log_prob = {random_log_prob:.2f}")
print(f"  (Should be a LARGE negative number, like -500 to -1000)")

# ============================================================
# Test with relaxed sigma for better initial sampling
# ============================================================

print("\n" + "="*70)
print("TESTING WITH RELAXED SIGMA")
print("="*70)

relaxed_nuisance = {'AA': 50.0, 'AB': 50.0, 'BC': 50.0}

random_log_prob_relaxed = log_probability_from_scores(
    random_flat.ravel(),
    random_radii,
    target_dists={'AA': 48.22, 'AB': 38.5, 'BC': 34.0},
    nuisance_parameters=relaxed_nuisance,
    unflatten_fn=unflatten_small
)

print(f"\nRandom structure (σ=50):")
print(f"  log_prob = {random_log_prob_relaxed:.2f}")
print(f"  (Should be less negative, maybe -20 to -50)")

# ============================================================
# Run MCMC with corrected function
# ============================================================

print("\n" + "="*70)
print("SETTING UP MCMC FOR SMALL SYSTEM")
print("="*70)

# Generate well-separated initial config
init_key = random.PRNGKey(42)
init_coords = generate_random_coords(
    key=init_key,
    n_dict={'A': 2, 'B': 2, 'C': 4},
    minval=-200.0,
    maxval=200.0
)

flat_coords_init, flat_radii = get_flat_coords_and_radii(init_coords, radii)
n_particles = flat_radii.shape[0]

# Start with relaxed sigma
nuisance_start = {'AA': 5.0, 'AB': 3.0, 'BC': 4.0}

# Create log_prob function with correct unflatten
unflatten_small = create_unflatten_function({'A': 2, 'B': 2, 'C': 4})

log_prob_fn = lambda coords_flat: log_probability_from_scores(
    coords_flat,
    flat_radii,
    target_dists={'AA': 48.22, 'AB': 38.5, 'BC': 34.0},
    nuisance_parameters=nuisance_start,
    unflatten_fn=unflatten_small,
    k_ev=1.0
)

# Test acceptance rates
print("\nTesting step sizes:")
print("   Step Size  |  Initial log_prob  |  Acceptance")
print("   " + "-"*55)

initial_log_prob = log_prob_fn(flat_coords_init.ravel())
print(f"   Initial    |  {initial_log_prob:15.2f}  |   -")

for test_step in [0.5, 1.0, 2.0, 5.0, 10.0]:
    test_kernel = setup_coordinate_rmh(log_prob_fn, test_step)
    test_key = random.PRNGKey(999)
    
    test_positions, test_log_probs, test_accepts = run_rmh_chain(
        test_key, test_kernel, flat_coords_init.ravel(), 1000
    )
    
    acc_rate = float(jnp.mean(test_accepts))
    final_log_prob = float(test_log_probs[-1])
    
    print(f"   {test_step:8.1f}   |  {final_log_prob:15.2f}  |  {acc_rate:5.1%}")

# Choose step size with ~20-40% acceptance
optimal_step = 15.0

print(f"\n{'='*70}")
print(f"RUNNING MCMC")
print(f"{'='*70}")
print(f"Using step_size = {optimal_step}")

rmh_kernel = setup_coordinate_rmh(log_prob_fn, optimal_step)

positions, log_probs, acc_rate = run_rmh_chain_with_logging(
    key=random.PRNGKey(456),
    kernel=rmh_kernel,
    initial_position=flat_coords_init.ravel(),
    n_steps=500_000,
    save_every=500,
    print_every=2500,
    output_file="mcmc_coords_correct.h5"
)

print(f"\n{'='*70}")
print(f"RESULTS")
print(f"{'='*70}")
print(f"Acceptance rate: {acc_rate:.1%}")
print(f"Initial log_prob: {log_probs[0]:.2f}")
print(f"Final log_prob: {log_probs[-1]:.2f}")
print(f"Best log_prob: {np.max(log_probs):.2f}")
print(f"Improvement: {float(np.max(log_probs) - log_probs[0]):.2f}")

# Verify final config is better than initial
if np.max(log_probs) > log_probs[0]:
    print("\n✓ MCMC improved the configuration!")
else:
    print("\n⚠ MCMC did not improve - try larger step size or more steps")

VERIFYING SCORE CONVERSION

System size: 32 particles (96 coordinates)

Ideal structure (σ=1):
  log_prob = -43.19
  (Should be a SMALL negative number, like -10 to -50)

Small system: 8 particles (24 coordinates)

Random structure (σ=1):
  log_prob = -234169.33
  (Should be a LARGE negative number, like -500 to -1000)

TESTING WITH RELAXED SIGMA

Random structure (σ=50):
  log_prob = -146.80
  (Should be less negative, maybe -20 to -50)

SETTING UP MCMC FOR SMALL SYSTEM

Testing step sizes:
   Step Size  |  Initial log_prob  |  Acceptance
   -------------------------------------------------------
   Initial    |        -19327.33  |   -
        0.5   |        -19327.33  |   0.0%
        1.0   |        -19327.33  |   0.0%
        2.0   |        -19327.33  |   0.0%
        5.0   |         -9768.86  |   0.8%
       10.0   |         -1393.38  |   0.7%

RUNNING MCMC
Using step_size = 15.0
Starting MCMC Sampling
Total steps: 500,000
Save every: 500
Initial log_prob: -19327.3281

Step       0